# Simple RAG (Retrieval-Augmented Generation) System

## Overview

This notebook implements a basic RAG pipeline for querying a PDF document using:

- `pypdf` for PDF parsing
- Google Gemini embeddings (`models/text-embedding-001`) for vectorization
- `faiss` for similarity search
- Gemini generation (`gemini-3-flash-preview`) for final answer synthesis

## Key Steps

1. Load and parse the PDF
2. Split text into overlapping chunks
3. Build a FAISS index from Gemini embeddings
4. Retrieve top-k chunks for a user question
5. Generate a grounded answer using retrieved context

## Environment

The notebook loads `GEMINI_API_KEY` from a local `.env` file via `python-dotenv`.

# Overall Summary

![overall](https://raw.githubusercontent.com/shubhra-opensource/rag-systems-cookbook/refs/heads/main/01_Jupyter_Notebooks/images/01.png)

# Package Installation and Imports

The cell below installs all necessary packages required to run this notebook.


In [ ]:
# Install required packages
# !pip install pypdf==5.6.0
# !pip install PyMuPDF==1.26.1
# !pip install python-dotenv==1.1.0
# !pip install faiss-cpu==1.11.0
# !pip install google-generativeai==0.8.5
# !pip install numpy==2.2.6

In [ ]:
import os
import numpy as np
import faiss
from pypdf import PdfReader
from dotenv import load_dotenv
import google.generativeai as genai

In [ ]:
# Load environment variables from local .env file
load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY not found. Add it to your .env file.")
genai.configure(api_key=GEMINI_API_KEY)
GENERATION_MODEL = "gemini-3-flash-preview"

In [ ]:
def replace_t_with_space(chunks):
    return [chunk.replace("\t", " ") for chunk in chunks]


def show_context(context_chunks):
    for i, chunk in enumerate(context_chunks, start=1):
        print(f"Context {i}:\n{chunk}\n")


### Read Docs

In [ ]:
# Download required data files
import os
os.makedirs('data', exist_ok=True)
path = "data/Understanding_Climate_Change.pdf"

### Encode document

In [ ]:
def _extract_pdf_text(path):
    reader = PdfReader(path)
    pages = [page.extract_text() or "" for page in reader.pages]
    return "\n".join(pages)
    # return pages
full_text=_extract_pdf_text(path)
full_text

In [ ]:
def _chunk_text(text, chunk_size=1000, chunk_overlap=200):
    chunks = []
    start = 0
    text_len = len(text)

    while start < text_len:
        end = min(start + chunk_size, text_len)
        chunks.append(text[start:end])

        if end == text_len:
            break

        start = max(end - chunk_overlap, 0)

    return chunks
chunk_size=1000
chunk_overlap=200
chunks = _chunk_text(full_text, chunk_size=chunk_size, chunk_overlap=chunk_overlap)
chunks

In [ ]:
chunks = replace_t_with_space(chunks)

In [ ]:
def _embed_texts(texts, task_type):
    vectors = []
    for text in texts:
        response = genai.embed_content(
            model="models/gemini-embedding-001",
            content=text,
            task_type=task_type,
        )
        vectors.append(response["embedding"])

    vectors = np.array(vectors, dtype="float32")
    faiss.normalize_L2(vectors)
    return vectors

In [ ]:
doc_embeddings = _embed_texts(chunks, task_type="retrieval_document")

In [ ]:
index = faiss.IndexFlatIP(doc_embeddings.shape[1])
index.add(doc_embeddings)

In [ ]:
def encode_pdf(path, chunk_size=1000, chunk_overlap=200):
    """
    Encodes a PDF book into a FAISS index using Gemini embeddings.

    Returns:
        Dict with FAISS index and source chunks.
    """
    full_text = _extract_pdf_text(path)
    chunks = _chunk_text(full_text, chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    chunks = replace_t_with_space(chunks)

    doc_embeddings = _embed_texts(chunks, task_type="retrieval_document")

    index = faiss.IndexFlatIP(doc_embeddings.shape[1])
    index.add(doc_embeddings)

    return {
        "index": index,
        "chunks": chunks,
    }

In [ ]:
def retrieve_context_per_question(question, vector_store, k=2):
    query_vector = _embed_texts([question], task_type="retrieval_query")
    _, indices = vector_store["index"].search(query_vector, k)

    return [vector_store["chunks"][i] for i in indices[0] if i != -1]


def answer_with_gemini(question, contexts):
    context_block = "\n\n---\n\n".join(contexts)
    prompt = f"""
You are a helpful assistant. Answer the question using only the context below.
If the answer is not in the context, say so clearly.

Context:
{context_block}

Question:
{question}
"""

    model = genai.GenerativeModel(GENERATION_MODEL)
    response = model.generate_content(prompt)
    return response.text

In [ ]:
chunks_vector_store = encode_pdf(path, chunk_size=1000, chunk_overlap=200)

### Create retriever

In [ ]:
top_k = 2

### Test retriever

In [ ]:
test_query = "What is the main cause of climate change?"
context = retrieve_context_per_question(test_query, chunks_vector_store, k=top_k)
show_context(context)

### Evaluate results

In [ ]:
answer = answer_with_gemini(test_query, context)
print(answer)

![](https://europe-west1-rag-techniques-views-tracker.cloudfunctions.net/rag-techniques-tracker?notebook=all-rag-techniques--simple-rag)